# 04_10 Clustering - BisectingKMeans
Train and evaluate BisectingKMeans.

[COMMAND_SO]
Command 1

[COMMAND_MUC_DICH]
- Muc tieu nghiep vu: Train BisectingKMeans va xem phan bo cluster de so sanh voi KMeans.
- Muc tieu ky thuat: Hien thi bang metric va bieu do cluster size ngay tren notebook.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.ml.clustering import BisectingKMeans
from pyspark.ml.evaluation import ClusteringEvaluator
spark=(SparkSession.builder.appName('04_10_bkm').master('local[2]').config('spark.sql.shuffle.partitions','16').getOrCreate())
spark.sparkContext.setLogLevel('WARN')
PROJECT_ROOT=Path.cwd().resolve().parent if Path.cwd().name=='notebooks' else Path.cwd().resolve()
FEATURE_DIR=PROJECT_ROOT/'data'/'processed'/'features'
MODEL_DIR=PROJECT_ROOT/'models'/'clustering'/'bisecting_kmeans'
METRIC_DIR=PROJECT_ROOT/'reports'/'model_metrics'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
train_df=spark.read.parquet(str(FEATURE_DIR/'clustering_train')).select('customer_unique_id','features').dropna()
val_df=spark.read.parquet(str(FEATURE_DIR/'clustering_val')).select('customer_unique_id','features').dropna()
test_df=spark.read.parquet(str(FEATURE_DIR/'clustering_test')).select('customer_unique_id','features').dropna()
bkm=BisectingKMeans(featuresCol='features',predictionCol='prediction',k=6,maxIter=50,seed=42)
m=bkm.fit(train_df)
pred_val=m.transform(val_df)
pred_test=m.transform(test_df)
sil_val=ClusteringEvaluator(featuresCol='features',predictionCol='prediction',metricName='silhouette').evaluate(pred_val)
sil_test=ClusteringEvaluator(featuresCol='features',predictionCol='prediction',metricName='silhouette').evaluate(pred_test)
metrics={'model_family':'clustering','model_name':'BisectingKMeans','val_silhouette':float(sil_val),'test_silhouette':float(sil_test),'train_rows':train_df.count(),'val_rows':val_df.count(),'test_rows':test_df.count(),'k':6}
print(metrics)
display(pd.DataFrame([metrics]))
cluster_pdf=pred_test.groupBy('prediction').count().orderBy('prediction').toPandas()
if not cluster_pdf.empty:
    display(cluster_pdf)
    plt.figure(figsize=(7,4))
    plt.bar(cluster_pdf['prediction'].astype(str), cluster_pdf['count'])
    plt.title('BisectingKMeans Cluster Size Distribution (test)')
    plt.xlabel('Cluster')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
m.write().overwrite().save(str(MODEL_DIR))
(METRIC_DIR/'clustering_bisecting_kmeans.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 00:30:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/04/03 00:30:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/03 00:30:03 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/03 00:30:03 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/03 00:30:03 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


26/04/03 00:30:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/03 00:30:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


{'model_family': 'clustering', 'model_name': 'BisectingKMeans', 'val_silhouette': 0.4096562018097793, 'test_silhouette': 0.4108063954164002, 'train_rows': 65592, 'val_rows': 13799, 'test_rows': 13967, 'k': 6}


,model_family,model_name,val_silhouette,test_silhouette,train_rows,val_rows,test_rows,k
0,clustering,BisectingKMeans,0.409656,0.410806,65592,13799,13967,6


,prediction,count
0,0,6816
1,1,863
2,2,297
3,3,3020
4,4,2184
5,5,787


/var/folders/07/jh7lmpq57r72h5jdjy3vwy740000gn/T/ipykernel_98797/2976324841.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


226